# Модель M/M/c (Система массового обслуживания)

**Цель работы:** Исследовать систему массового обслуживания с несколькими
каналами, изучить зависимость времени ожидания от загрузки системы.

## 1. Теоретическое введение

Модель M/M/c характеризуется:
* Пуассоновским входящим потоком с интенсивностью λ
* Экспоненциальным распределением времени обслуживания с интенсивностью μ
* Количеством каналов c

## 2. Параметры модели
- λ = 0.9 заявок/ед. времени
- μ = 0.5 заявок/ед. времени
- c = 2 канала

## 3. Результаты
scripts/mmc_run.jl
Запуск модели M/M/c и построение графиков

In [1]:
using DrWatson
@quickactivate "project"

using Plots
using DataFrames
using Statistics
using LaTeXStrings

include(srcdir("mmc.jl"))

run_mmc_stats (generic function with 1 method)

-------------------------------------------------------------------
1. БАЗОВЫЙ ЗАПУСК
-------------------------------------------------------------------

In [2]:
println("="^60)
println("МОДЕЛЬ M/M/c")
println("="^60)

print("Запуск базовой симуляции... ")
sim_time = run_mmc(λ=0.9, μ=0.5, c=2, n_customers=20)
println("завершено за $sim_time ед. времени")

МОДЕЛЬ M/M/c
Запуск базовой симуляции... [1.3064129818164971] Клиент 1 прибыл
[1.3064129818164971] Клиент 1 начал обслуживание
[1.9865651038277927] Клиент 1 завершил обслуживание
[3.2607291700959573] Клиент 2 прибыл
[3.2607291700959573] Клиент 2 начал обслуживание
[4.381131796462752] Клиент 3 прибыл
[4.381131796462752] Клиент 3 начал обслуживание
[4.7712033870811155] Клиент 4 прибыл
[5.476566325272271] Клиент 5 прибыл
[6.788898448990592] Клиент 6 прибыл
[6.839393681370896] Клиент 7 прибыл
[7.505451139077112] Клиент 8 прибыл
[8.768843171514131] Клиент 2 завершил обслуживание
[8.768843171514131] Клиент 4 начал обслуживание
[9.069440548209927] Клиент 4 завершил обслуживание
[9.069440548209927] Клиент 5 начал обслуживание
[9.097413728753494] Клиент 9 прибыл
[9.195251078847564] Клиент 5 завершил обслуживание
[9.195251078847564] Клиент 6 начал обслуживание
[9.669962665292429] Клиент 6 завершил обслуживание
[9.669962665292429] Клиент 7 начал обслуживание
[9.779255725435968] Клиент 7 завершил 

-------------------------------------------------------------------
2. СБОР СТАТИСТИКИ
-------------------------------------------------------------------

In [3]:
print("\nСбор статистики... ")
stats, sim_time = run_mmc_stats(λ=0.9, μ=0.5, c=2, n_customers=100)
println("завершено")

wait_times = collect(values(stats[:wait_times]))
service_times = collect(values(stats[:service_times]))

println("\n📊 Статистика:")
println("  Среднее время ожидания в очереди: W_q = ", round(mean(wait_times), digits=4))
println("  Среднее время обслуживания: 1/μ = ", round(mean(service_times), digits=4))
println("  Среднее время в системе: W = ", round(mean(wait_times .+ service_times), digits=4))


Сбор статистики... завершено

📊 Статистика:
  Среднее время ожидания в очереди: W_q = 4.9805
  Среднее время обслуживания: 1/μ = 1.8796
  Среднее время в системе: W = 6.8602


-------------------------------------------------------------------
3. ПОСТРОЕНИЕ ГРАФИКОВ
-------------------------------------------------------------------

In [4]:
script_name = splitext(basename(PROGRAM_FILE))[1]
mkpath(plotsdir(script_name))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab07/project/plots"

Гистограмма времени ожидания

In [5]:
p1 = histogram(wait_times,
    bins=20,
    xlabel="Время ожидания в очереди",
    ylabel="Частота",
    title="Распределение времени ожидания (M/M/2)",
    legend=false,
    color=:blue,
    alpha=0.7
)
savefig(p1, plotsdir(script_name, "mmc_wait_hist.png"))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab07/project/plots/mmc_wait_hist.png"

Параметрическое исследование: влияние загрузки

In [6]:
loads = 0.3:0.1:0.9
ρ = loads
wait_times_mean = Float64[]

for ρ_val in loads
    λ_val = ρ_val * 2 * 0.5  # λ = ρ * c * μ
    stats, _ = run_mmc_stats(λ=λ_val, μ=0.5, c=2, n_customers=200)
    push!(wait_times_mean, mean(collect(values(stats[:wait_times]))))
end

p2 = plot(loads, wait_times_mean,
    marker=:circle,
    xlabel="Загрузка системы ρ = λ/(c·μ)",
    ylabel="Среднее время ожидания W_q",
    title="Зависимость времени ожидания от загрузки",
    linewidth=2,
    legend=false
)
savefig(p2, plotsdir(script_name, "mmc_load_vs_wait.png"))

println("\n✅ Графики сохранены в: ", plotsdir(script_name))


✅ Графики сохранены в: /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab07/project/plots/
